# Zero-cost collar, implied forward and implied repo
Three uses of vanilla prices: protect a stock position for free with a collar, read the forward off put–call parity,
and the repo implied in that forward (Exercises, Equity: Forwards and dividends, Options and Black–Scholes).

In [ ]:
import os
from datetime import date, timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq
import tensorquant as tq

## 1. Zero-cost collar
Bought at 15, the stock is at 19. To be sure not to end below 15 in one year: **buy the 1Y put struck at 15** (floor)
and pay for it by **selling a 1Y call** struck at $K_c$ chosen so that $C(K_c) = P(15)$. At expiry the position is worth
$\min(\max(S_T, 15), K_c)$: losses below 15 are covered, the upside above $K_c$ is given away, nothing is paid today.
A hypothetical stock: no dividends, flat 2.5% rate, flat 30% vol.

In [ ]:
TODAY = date(2026, 11, 5)
tq.Settings.evaluation_date = TODAY
EXPIRY = TODAY + timedelta(days=365)
S0, PAID = 19.0, 15.0


def stock_market(vol_surface):
    return tq.MarketEnvironment({"IR:EUR:ESTR:SPOT": tq.FlatCurve(TODAY, 0.025, tq.DayCounterConvention.Actual365),
                                 "EQ:EUR:STOCK:SPOT": S0, "EQ:EUR:STOCK:VOL": vol_surface})


def bs(mkt, option_type, strike):
    option = tq.VanillaOption(tq.Currency.EUR, TODAY, EXPIRY, option_type, strike, underlying="STOCK")
    tq.BlackScholesPricer().price(option, mkt)
    return float(option.price)


def zero_cost_call_strike(mkt):
    put = bs(mkt, tq.OptionType.Put, PAID)
    return put, brentq(lambda k: bs(mkt, tq.OptionType.Call, k) - put, S0, 5 * S0)   # the call price falls with K


flat = stock_market(tq.BlackConstantVolatility(TODAY, 0.30))
put, k_call = zero_cost_call_strike(flat)
print(f"put 15: {put:.4f}   zero-cost call strike: {k_call:.2f} ({k_call / S0:.0%} of spot)   "
      f"call({k_call:.2f}) = {bs(flat, tq.OptionType.Call, k_call):.4f}")

In [ ]:
s = np.linspace(5, 40, 400)
collar = np.minimum(np.maximum(s, PAID), k_call)
plt.figure(figsize=(7, 4))
plt.plot(s, s, "--", color="grey", label="stock only")
plt.plot(s, collar, lw=2, label=f"stock + long put {PAID:g} + short call {k_call:.2f}")
plt.axhline(PAID, color="k", lw=0.8, ls=":")
plt.xlabel("$S_T$"); plt.ylabel("value at expiry"); plt.legend(); plt.title("Zero-cost collar");

On an equity skew the put (out of the money downside) is priced at a higher vol than the call: the put costs more, so
the call to sell must be closer to the money and **more upside** is given away. Same stock, a skewed surface
(vol 36% at 15, 30% at 19, 27% at 25 and above):

In [ ]:
skew = tq.VolatilitySurface(reference_date=TODAY, calendar=None,
                            daycounter=tq.DayCounter(tq.DayCounterConvention.Actual365),
                            strike=[10.0, 15.0, 19.0, 25.0, 40.0], maturity=[0.5, 2.0],
                            volatility_matrix=[[0.40, 0.36, 0.30, 0.27, 0.27]] * 2)
put_s, k_call_s = zero_cost_call_strike(stock_market(skew))
pd.DataFrame([{"surface": "flat 30%", "put 15": put, "call strike": k_call, "cap (% spot)": 100 * k_call / S0},
              {"surface": "skewed", "put 15": put_s, "call strike": k_call_s, "cap (% spot)": 100 * k_call_s / S0}]
             ).set_index("surface").round(3)

## 2. The forward implied by put–call parity
For European options on the same expiry, $C(K) - P(K) = P(0,T)\,(F - K)$: linear in $K$ with slope $-P(0,T)$ and
intercept $P(0,T)F$. With a call and a put at **one** strike and the discount factor known, $F = K + (C - P)/P(0,T)$;
with **two or more** strikes the line gives both: $P(0,T) = -\,$slope and $F = $ intercept $/ P(0,T)$ (in practice a fit
over the liquid strikes near the money). Here the "quotes" are Black–Scholes prices on the TICKER1 smile.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")       # latest snapshot
tq.Settings.evaluation_date = mkt.evaluation_date
eq = mkt.get_equity("TICKER1")
T_EXPIRY = mkt.evaluation_date + timedelta(days=365)
T = eq.year_fraction(T_EXPIRY)
pricer = tq.BlackScholesPricer(dividend_model="discrete")


def quote(option_type, strike):
    option = tq.VanillaOption(eq.currency, mkt.evaluation_date, T_EXPIRY, option_type, strike, underlying="TICKER1")
    pricer.price(option, mkt)
    return float(option.price)


strikes = float(eq.spot) * np.array([0.8, 0.9, 1.0, 1.1, 1.2])
quotes = pd.DataFrame({"K": strikes, "call": [quote(tq.OptionType.Call, k) for k in strikes],
                       "put": [quote(tq.OptionType.Put, k) for k in strikes]})
quotes["C - P"] = quotes["call"] - quotes["put"]
slope, intercept = np.polyfit(quotes["K"], quotes["C - P"], 1)
df_implied, f_implied = -slope, intercept / -slope
k = quotes.iloc[2]                                                 # one strike, discount factor from the curve
f_one = k["K"] + k["C - P"] / float(eq.discount_curve.discount(T))
display(quotes.round(4))
pd.DataFrame({"implied": [f_implied, df_implied, f_one], "market": [float(eq.forward(T)),
              float(eq.discount_curve.discount(T)), float(eq.forward(T))]},
             index=["forward (fit)", "discount factor (fit)", "forward (one strike)"]).round(6)

## 3. The repo implied in the forward
The equity forward is $F = (S_0 - \mathrm{PV}(\text{div}))\,e^{-q_{\text{repo}}T} / P(0,T)$: the stock bought today and held to
$T$ costs the funding ($1/P(0,T)$), returns the dividends and earns the **repo**, the fee from lending the stock (a
short seller must borrow it and pay that fee). Solving for it:
$$q_{\text{repo}} = -\frac{1}{T}\ln\frac{F\,P(0,T)}{S_0 - \mathrm{PV}(\text{div})}.$$
With the market forward (or the one implied from the options) and the dividends known, the repo comes out; in practice
the implied "repo" also absorbs the error in the dividend forecast and any funding spread, so desks often quote an
**implied dividend / repo** pair from the option market.

In [ ]:
s0, pv_div, df = float(eq.spot), float(eq.pv_dividends(T)), float(eq.discount_curve.discount(T))
repo_implied = -np.log(f_implied * df / (s0 - pv_div)) / T
print(f"implied repo {100 * repo_implied:.4f}%   market repo {100 * float(eq.repo):.4f}%")

In [ ]:
# the same at every maturity of the surface: one implied repo per expiry (here flat, as in the snapshot)
rows = []
for t in [float(x) for x in eq.vol_surface.maturity]:
    expiry = mkt.evaluation_date + timedelta(days=int(round(365 * t)))
    tt = eq.year_fraction(expiry)
    ks = float(eq.forward(tt)) * np.array([0.95, 1.0, 1.05])
    cp = []
    for kk in ks:
        c = tq.VanillaOption(eq.currency, mkt.evaluation_date, expiry, tq.OptionType.Call, kk, underlying="TICKER1")
        p = tq.VanillaOption(eq.currency, mkt.evaluation_date, expiry, tq.OptionType.Put, kk, underlying="TICKER1")
        pricer.price(c, mkt); pricer.price(p, mkt)
        cp.append(float(c.price) - float(p.price))
    b, a = np.polyfit(ks, cp, 1)
    f, d = a / -b, -b
    rows.append({"expiry": expiry, "T": tt, "implied forward": f,
                 "implied repo (%)": -100 * np.log(f * d / (s0 - float(eq.pv_dividends(tt)))) / tt})
pd.DataFrame(rows).set_index("expiry").round(4)

## 4. American options: parity bounds and the implied forward
Early exercise breaks the equality: with dividends of present value $D$,
$$S_0 - D - K \;\le\; C_A - P_A \;\le\; S_0 - K\,P(0,T).$$
Upper bound: the put can be exercised early, so $P_A \ge P_E$, while the call gains from early exercise only to
capture a dividend; lower bound: a long call plus $K + D$ in cash is worth at least a long put plus the stock, whenever
the put is exercised. The European $C_E - P_E$ lies inside. A hand-built stock: spot 100, one dividend of 2 in six
months, flat 3% rate, flat 25% vol, American prices by Bjerksund–Stensland on the spot net of the dividends.

In [ ]:
AM_TODAY = date(2026, 11, 5)
tq.Settings.evaluation_date = AM_TODAY
AM_EXPIRY = AM_TODAY + timedelta(days=365)
S_AM, DIV_AM = 100.0, 2.0
am_mkt = tq.MarketEnvironment({
    "IR:EUR:ESTR:SPOT": tq.FlatCurve(AM_TODAY, 0.03, tq.DayCounterConvention.Actual365),
    "EQ:EUR:AM:SPOT": S_AM, "EQ:EUR:AM:REPO": 0.0,
    "EQ:EUR:AM:DIV": tq.DividendCurve(reference_date=AM_TODAY, ex_dates=[AM_TODAY + timedelta(days=182)],
                                      amounts=[DIV_AM], currency=tq.Currency.EUR,
                                      daycounter_convention=tq.DayCounterConvention.Actual365),
    "EQ:EUR:AM:VOL": tq.BlackConstantVolatility(AM_TODAY, 0.25)})
am = am_mkt.get_equity("AM")
am_pricer = tq.BlackScholesPricer(dividend_model="discrete")
T_AM = am.year_fraction(AM_EXPIRY)
DF_AM = float(am.discount_curve.discount(T_AM))


def am_price(option_type, strike, exercise=tq.ExerciseType.American):
    o = tq.VanillaOption(tq.Currency.EUR, AM_TODAY, AM_EXPIRY, option_type, strike, underlying="AM",
                         exercise_type=exercise)
    am_pricer.price(o, am_mkt)
    return float(o.price)


rows = []
for k in (80.0, 90.0, 100.0, 110.0, 120.0):
    rows.append({"K": k, "S - D - K": S_AM - DIV_AM - k,
                 "C_A - P_A": am_price(tq.OptionType.Call, k) - am_price(tq.OptionType.Put, k),
                 "S - K P(0,T)": S_AM - k * DF_AM,
                 "C_E - P_E": am_price(tq.OptionType.Call, k, tq.ExerciseType.European)
                 - am_price(tq.OptionType.Put, k, tq.ExerciseType.European),
                 "put early-exercise premium": am_price(tq.OptionType.Put, k)
                 - am_price(tq.OptionType.Put, k, tq.ExerciseType.European)})
pd.DataFrame(rows).set_index("K").round(4)

**The forward from American quotes.** $F = K + (C_A - P_A)/P(0,T)$ is wrong: the early-exercise premia (mostly the
put's) bias it. The standard way: the forward is a function of the unknown repo (dividends and rates known); for a
trial repo, imply the vol from the American call and from the American put at the same strike with the American
model; the right repo is the one at which **the two implied vols coincide** (one forward for both, one vol for both).
Below the "quotes" are American prices with a 1% repo; the solver recovers it.

In [ ]:
TRUE_REPO, K_AM = 0.01, 100.0
am.repo.assign(TRUE_REPO)
quote_call, quote_put = am_price(tq.OptionType.Call, K_AM), am_price(tq.OptionType.Put, K_AM)
true_forward = float(am.forward(T_AM))
flat_vol = am.vol_surface.flat_vol


def american_implied_vol(quote, option_type):
    def gap(v):
        flat_vol.assign(v)
        return am_price(option_type, K_AM) - quote
    return brentq(gap, 0.05, 1.5)


def vol_gap(repo):
    am.repo.assign(repo)
    return american_implied_vol(quote_call, tq.OptionType.Call) - american_implied_vol(quote_put, tq.OptionType.Put)


naive_forward = K_AM + (quote_call - quote_put) / DF_AM
repo_hat = brentq(vol_gap, -0.03, 0.05)
am.repo.assign(repo_hat)
pd.DataFrame({"value": [true_forward, naive_forward, float(am.forward(T_AM)), 100 * repo_hat,
                        100 * american_implied_vol(quote_call, tq.OptionType.Call)]},
             index=["true forward", "European parity on American prices", "forward (equal implied vols)",
                    "implied repo (%)", "implied vol (%)"]).round(4)

## 5. Conversion and reversal
**Conversion**: buy the stock, buy the put, sell the call (same $K$, $T$): at expiry the stock is delivered at $K$
whatever happens. It costs $S_0 - C + P$ today, earns the dividends and the repo (the stock can be lent), and returns
$K$ at $T$. Its fair value is zero when $C - P = P(0,T)(F - K)$; if the call is rich, the conversion locks in the
excess. **Reversal**: the opposite (short stock, sell the put, buy the call); the short seller pays the dividends and
the **repo**. Same stock as section 4, repo 1%, call quoted 0.40 above its fair value:

In [ ]:
am.repo.assign(0.01)
flat_vol.assign(0.25)
K_C = 100.0
call_fair = am_price(tq.OptionType.Call, K_C, tq.ExerciseType.European)
put_fair = am_price(tq.OptionType.Put, K_C, tq.ExerciseType.European)
call_quote = call_fair + 0.40
forward = float(am.forward(T_AM))
conversion_pv = (call_quote - put_fair) - DF_AM * (forward - K_C)
print(f"forward {forward:.4f}   fair C - P {call_fair - put_fair:.4f} = P(0,T)(F - K) {DF_AM * (forward - K_C):.4f}")
print(f"quoted C - P {call_quote - put_fair:.4f}  ->  conversion locks {conversion_pv:.4f} today")

The repo matters on the reversal side: a trader who forgets the borrow fee computes the forward with zero repo and
sees the fair $C - P$ as "too low", a fake arbitrage worth exactly the fee he would pay to borrow the stock.

In [ ]:
am.repo.assign(0.0)
forward_no_repo = float(am.forward(T_AM))
am.repo.assign(0.01)
fake = DF_AM * (forward_no_repo - forward)
print(f"forward without repo {forward_no_repo:.4f} vs {forward:.4f}: apparent reversal profit {fake:.4f} "
      f"= PV of the borrow fee")

## 6. Box spread
Long call $K_1$, short call $K_2$, long put $K_2$, short put $K_1$ (a bull call spread plus a bear put spread): at
expiry it pays $K_2 - K_1$ whatever $S_T$, so its price is $(K_2 - K_1)\,P(0,T)$, independent of spot, dividends,
repo and vol. Box quotes give the rate implied in the option market, $r = -\ln(\text{box}/(K_2 - K_1))/T$. With
American options the short puts can be exercised early: the box is no longer riskless.

In [ ]:
def ticker_option(option_type, strike, exercise=tq.ExerciseType.European):
    o = tq.VanillaOption(eq.currency, mkt.evaluation_date, T_EXPIRY, option_type, strike, underlying="TICKER1",
                         exercise_type=exercise)
    pricer.price(o, mkt)
    return float(o.price)


def box(k1, k2, exercise=tq.ExerciseType.European):
    return (ticker_option(tq.OptionType.Call, k1, exercise) - ticker_option(tq.OptionType.Call, k2, exercise)
            + ticker_option(tq.OptionType.Put, k2, exercise) - ticker_option(tq.OptionType.Put, k1, exercise))


k1, k2 = 0.9 * float(eq.spot), 1.1 * float(eq.spot)
tq.Settings.evaluation_date = mkt.evaluation_date
b_eu, b_am = box(k1, k2), box(k1, k2, tq.ExerciseType.American)
pd.DataFrame({"value": [b_eu, (k2 - k1) * float(eq.discount_curve.discount(T)),
                        -100 * np.log(b_eu / (k2 - k1)) / T, 100 * float(eq.discount_curve.zero_rate(T)), b_am]},
             index=["European box", "(K2 - K1) P(0,T)", "implied rate (%)", "curve zero rate (%)",
                    "American box"]).round(4)

## 7. A call and a put with the same strike
Differentiate parity, $C - P = P(0,T)(F - K)$: the right side has no vol and is linear in the spot, so the call and
the put have the **same vega and the same gamma**, and $\Delta_C - \Delta_P = P(0,T)\,\partial F/\partial S = e^{-q_{\text{repo}}T}$
(dividends as cash amounts do not scale with the spot). By AAD on TICKER1, 1Y, strike at the money:

In [ ]:
import tensorflow as tf
spot_var = eq.spot


def greeks(option_type, strike):
    o = tq.VanillaOption(eq.currency, mkt.evaluation_date, T_EXPIRY, option_type, strike, underlying="TICKER1")
    with tf.GradientTape() as outer:
        with tf.GradientTape() as inner:
            price = pricer.calculate_price(o, mkt)
        delta = inner.gradient(price, spot_var)
    gamma = outer.gradient(delta, spot_var)
    with tf.GradientTape() as tape:
        price = pricer.calculate_price(o, mkt)
    vega = float(tf.reduce_sum(tape.gradient(price, eq.vol_surface.volatility_matrix))) / 100
    return {"price": float(price), "delta": float(delta), "gamma": float(gamma), "vega (1 vol pt)": vega}


tq.Settings.evaluation_date = mkt.evaluation_date
k_atm = round(float(eq.forward(T)), 2)
g = pd.DataFrame({"call": greeks(tq.OptionType.Call, k_atm), "put": greeks(tq.OptionType.Put, k_atm)})
g["call - put"] = g["call"] - g["put"]
print(f"exp(-repo T) = {np.exp(-float(eq.repo) * T):.6f}")
g.round(6)

## 8. Early exercise of an American call before an ex-dividend date
Without dividends an American call is never exercised early: $C_A \ge C_E \ge S - K\,P(0,T) > S - K$. With a dividend
$D$ at $t_d$, just before the ex-date the holder compares exercising now ($S - K$) with keeping a call on the stock
after the drop ($S - D$, no more dividends to expiry, so the American call is worth the European one). It exercises
when $D$ exceeds the interest on the strike plus the time value it gives up; a necessary condition is
$D > K\,(1 - P(t_d, T))$. Stock at 120 just before a dividend of 2 or of 0.5, three months to expiry, 3% rate, 25% vol:

In [ ]:
def call_after_drop(spot, strike, dividend, days_left=91, vol=0.25, rate=0.03):
    day = date(2026, 11, 5)
    tq.Settings.evaluation_date = day
    m = tq.MarketEnvironment({"IR:EUR:ESTR:SPOT": tq.FlatCurve(day, rate, tq.DayCounterConvention.Actual365),
                              "EQ:EUR:X:SPOT": spot - dividend, "EQ:EUR:X:VOL": tq.BlackConstantVolatility(day, vol)})
    o = tq.VanillaOption(tq.Currency.EUR, day, day + timedelta(days=days_left), tq.OptionType.Call, strike,
                         underlying="X")
    tq.BlackScholesPricer().price(o, m)
    df = float(m.get_ir_curve(tq.Currency.EUR).discount(day + timedelta(days=days_left)))
    return float(o.price), strike * (1 - df)


rows = []
for dividend in (2.0, 0.5):
    for strike in (80.0, 100.0, 110.0):
        keep, interest = call_after_drop(120.0, strike, dividend)
        rows.append({"dividend": dividend, "strike": strike, "exercise now S - K": 120.0 - strike,
                     "keep (call after the drop)": keep, "K (1 - P)": interest,
                     "exercise?": "yes" if 120.0 - strike > keep else "no"})
pd.DataFrame(rows).set_index(["dividend", "strike"]).round(4)

## Exercises
1. Make the collar cost-neutral with a put struck at 17 instead of 15: how much upside is left?
2. Add a 2% noise to the option quotes of section 2: how does the error on the implied forward change with the number
   of strikes in the fit?
3. Raise the 1Y dividend by 1 and recompute the implied repo from the same forward: what does the repo absorb?
4. Repeat section 4 with a put struck at 120 (deep in the money): how large is the error of the European parity, and why?
